# DAA Kaggle Submission Notebook V.3

This notebook is TRYING to trains a multiclass model and TRYING to write `submission.csv` for the Kaggle DAA competition.

**IMPORTANT LOOK AT HERE AGAIN:** set `TRAIN_PATH` to the Kaggle training CSV. `dataset_test_X.csv` IS the provided test feature file. "IF" `sample_submission.csv` exists, the notebook uses its target column name and order.


In [ ]:

import os, warnings, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# ==== EDIT THESE PATHS IF NEEDED ====
TRAIN_PATH = "datasets/dataset_train.csv"              # e.g. "/kaggle/input/daa/train.csv"
TEST_PATH = "datasets/dataset_test_X.csv"      # provided file
SAMPLE_SUB_PATH = "sample_submission.csv"
SUB_PATH = "submission.csv"
TARGET = None                         # set manually if inference fails, like "age_group"
ID_COLS = ["PID", "HHID"]
# ====================================

test = pd.read_csv(TEST_PATH)
if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError("Set TRAIN_PATH to the Kaggle training CSV file.")
train = pd.read_csv(TRAIN_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH) if os.path.exists(SAMPLE_SUB_PATH) else None

# Infer target column
if TARGET is None:
    if sample_sub is not None:
        cand = [c for c in sample_sub.columns if c not in ID_COLS]
        if len(cand) == 1:
            TARGET = cand[0]
    if TARGET is None:
        for c in ["age_group", "AgeGroup", "AGE_GROUP", "age", "Age", "target", "Target", "class", "Class"]:
            if c in train.columns:
                TARGET = c
                break
    if TARGET is None:
        not_in_test = [c for c in train.columns if c not in test.columns and c not in ID_COLS]
        if len(not_in_test) == 1:
            TARGET = not_in_test[0]
    if TARGET is None:
        raise ValueError("Could not infer target column. Set TARGET manually.")

print("Target column:", TARGET)

# Features: columns present in both train and test, excluding IDs and target
feature_cols = [c for c in train.columns if c not in [TARGET] + ID_COLS and c in test.columns]
X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

y_raw = train[TARGET].astype(str).copy()
test_ids = test["PID"] if "PID" in test.columns else pd.Series(np.arange(len(test)), name="PID")

le = LabelEncoder()
y = le.fit_transform(y_raw)

cat_cols = [c for c in feature_cols if X[c].dtype == "object" or str(X[c].dtype).startswith("category")]
for c in cat_cols:
    X[c] = X[c].fillna("MISSING").astype(str)
    X_test[c] = X_test[c].fillna("MISSING").astype(str)

print("N features:", len(feature_cols), "| categorical:", len(cat_cols))


In [ ]:

# Prefer CatBoost if available; otherwise fallback to sklearn HistGradientBoosting
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

if HAS_CATBOOST:
    print("Using CatBoost")
    base = CatBoostClassifier(
        loss_function="MultiClass",
        random_state=RANDOM_STATE,
        verbose=False,
        cat_features=cat_cols
    )
    param_dist = {
        "iterations": [300, 600],
        "depth": [4, 6, 8],
        "learning_rate": [0.03, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 10],
    }
    search = RandomizedSearchCV(
        base, param_dist, n_iter=8, scoring="f1_macro",
        cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
    )
    search.fit(X, y)
    best_model = search.best_estimator_
    print("Best CV macro F1:", search.best_score_)
    pred_enc = np.asarray(best_model.predict(X_test)).astype(int).ravel()

else:
    print("CatBoost not available; using sklearn HistGradientBoosting fallback")
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OrdinalEncoder
    from sklearn.ensemble import HistGradientBoostingClassifier

    num_cols = [c for c in feature_cols if c not in cat_cols]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols),
        ],
        remainder="drop"
    )

    base = Pipeline(steps=[
        ("prep", preprocessor),
        ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))
    ])

    param_dist = {
        "clf__learning_rate": [0.03, 0.05, 0.1],
        "clf__max_iter": [200, 400],
        "clf__max_leaf_nodes": [15, 31, 63],
        "clf__min_samples_leaf": [10, 20, 40],
        "clf__l2_regularization": [0, 0.1, 1.0],
    }
    search = RandomizedSearchCV(
        base, param_dist, n_iter=10, scoring="f1_macro",
        cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
    )
    search.fit(X, y)
    best_model = search.best_estimator_
    print("Best CV macro F1:", search.best_score_)
    pred_enc = best_model.predict(X_test).astype(int).ravel()

pred = le.inverse_transform(pred_enc)
print("Prediction distribution:")
print(pd.Series(pred).value_counts().head())


In [ ]:

# Build submission
target_sub_col = TARGET

if sample_sub is not None:
    sub = sample_sub.copy()
    cand = [c for c in sub.columns if c not in ID_COLS]
    if len(cand) >= 1:
        target_sub_col = cand[0]

    if "PID" in sub.columns and "PID" in test.columns:
        pred_map = pd.DataFrame({"PID": test_ids.values, target_sub_col: pred})
        sub = sample_sub.copy()
        sub[target_sub_col] = sub["PID"].map(pred_map.set_index("PID")[target_sub_col])
    else:
        sub[target_sub_col] = pred
else:
    sub = pd.DataFrame({"PID": test_ids, target_sub_col: pred})

sub.to_csv(SUB_PATH, index=False)
print("Saved:", SUB_PATH, sub.shape)
print(sub.head())
